# LeetCode 3363 — Find the Maximum Number of Fruits Collected bằng Python

- Link: https://leetcode.com/problems/find-the-maximum-number-of-fruits-collected/
- Độ khó: **Hard**
- Chủ đề: **Quy hoạch động trên lưới**, nhận xét **đường đi bị ép** (trẻ thứ nhất chỉ đi đường chéo), tách thành hai DP **độc lập** trên hai tam giác, dùng **chuyển vị** để tái sử dụng code

Skeleton trên LeetCode (Python3):

```python
class Solution:
    def maxCollectedFruits(self, fruits: List[List[int]]) -> int:
```

## 1. Mục tiêu học

1. Nhận ra trẻ xuất phát `(0, 0)` phải đi **đúng** `n − 1` bước tới `(n−1, n−1)` ⇒ mỗi bước tăng cả hàng và cột ⇒ **bị ép đi đường chéo chính**.
2. Trẻ xuất phát `(0, n−1)` sau bước thứ `i` luôn ở **hàng `i`**; tối ưu là luôn ở **trên đường chéo** (`j > i`) — không giành quả với hai trẻ còn lại.
3. Trẻ xuất phát `(n−1, 0)` đối xứng qua đường chéo ⇒ dùng DP của trẻ 2 trên **ma trận chuyển vị**.
4. Viết DP theo hàng `O(n²)` với `−∞` cho ô không đến được.
5. Kiểm chứng bằng vét cạn mọi cặp đường đi (`n ≤ 5`).

## 2. Đề bài & ví dụ

Lưới `n × n`, ô `(i, j)` có `fruits[i][j]` quả. Ba trẻ xuất phát ở `(0,0)`, `(0,n−1)`, `(n−1,0)`, mỗi trẻ đi **đúng** `n − 1` bước tới `(n−1, n−1)`:

- trẻ 1: `(i+1, j+1)`, `(i+1, j)`, `(i, j+1)`;
- trẻ 2: `(i+1, j−1)`, `(i+1, j)`, `(i+1, j+1)`;
- trẻ 3: `(i−1, j+1)`, `(i, j+1)`, `(i+1, j+1)`.

Ô nhiều trẻ đi qua chỉ tính quả một lần. Tìm tổng quả lớn nhất.

**Ví dụ 1:** `[[1,2,3,4],[5,6,8,7],[9,10,11,12],[13,14,15,16]]` → `100`. **Ví dụ 2:** `[[1,1],[1,1]]` → `4`.

**Ràng buộc:** `2 ≤ n ≤ 1000`, `0 ≤ fruits[i][j] ≤ 1000`.

In [1]:
from typing import List
import itertools

examples = [([[1, 2, 3, 4], [5, 6, 8, 7], [9, 10, 11, 12], [13, 14, 15, 16]], 100), ([[1, 1], [1, 1]], 4)]
for f, exp in examples:
    print(len(f), 'x', len(f), '-> kỳ vọng', exp)

4 x 4 -> kỳ vọng 100
2 x 2 -> kỳ vọng 4


## 3. Kiến thức nền

### 3.1. Đường đi của trẻ 1 bị ép

Từ `(0, 0)` tới `(n−1, n−1)` cần tăng hàng `n − 1` và cột `n − 1`. Mỗi bước tăng mỗi tọa độ tối đa 1, có `n − 1` bước ⇒ mọi bước phải là `(+1, +1)` ⇒ đường chéo chính.

### 3.2. Trẻ 2 luôn ở hàng `i` sau `i` bước

Mọi bước của trẻ 2 tăng hàng đúng 1 ⇒ sau bước `i` ở hàng `i`; cột thay đổi `−1, 0, +1`. Để về `(n−1, n−1)` từ `(i, j)` cần `n − 1 − j ≤ n − 1 − i`… cụ thể cột phải đạt `n − 1` sau đúng `n − 1 − i` bước nữa ⇒ ô `(i, j)` khả thi khi `j ≥ i` … và ta sẽ chứng minh nên giữ `j > i` (trước đích).

In [2]:
def child2_paths(n):
    res = []
    for mv in itertools.product((-1, 0, 1), repeat=n - 1):
        i, j, p, ok = 0, n - 1, [(0, n - 1)], True
        for d in mv:
            i += 1; j += d
            if not 0 <= j < n:
                ok = False; break
            p.append((i, j))
        if ok and p[-1] == (n - 1, n - 1):
            res.append(p)
    return res

print('số đường của trẻ 2 với n = 4:', len(child2_paths(4)), '| ví dụ:', child2_paths(4)[:2])

số đường của trẻ 2 với n = 4: 4 | ví dụ: [[(0, 3), (1, 2), (2, 2), (3, 3)], [(0, 3), (1, 2), (2, 3), (3, 3)]]


## 4. Ý tưởng nền tảng / cách nghĩ trực giác

Ví dụ 1: trẻ 1 lấy đường chéo `1 + 6 + 11 + 16 = 34`. Trẻ 2 đi `(0,3) → (1,2) → (2,3) → (3,3)`: `4 + 8 + 12` (đích đã tính) `= 24`. Trẻ 3 đi `(3,0) → (3,1) → (3,2) → (3,3)`: `13 + 14 + 15 = 42`. Tổng `100`.

Vì sao trẻ 2 không nên xuống dưới đường chéo: ô trên đường chéo đã bị trẻ 1 lấy (đi vào không được thêm gì), và phần dưới đường chéo là "địa bàn" của trẻ 3 — nếu trẻ 2 lấn xuống, có thể đổi phần đường đó cho trẻ 3 mà tổng không giảm (kiểm chứng bằng vét cạn ở mục 12).

## 5. Cách 1 — Brute Force: mọi cặp đường đi của trẻ 2 và trẻ 3

In [3]:
def max_fruits_brute(f):
    n = len(f)
    P2 = child2_paths(n)
    P3 = [[(j, i) for i, j in p] for p in P2]          # trẻ 3 là chuyển vị của trẻ 2
    diag = {(i, i) for i in range(n)}
    best = 0
    for a in P2:
        for b in P3:
            cells = diag | set(a) | set(b)
            best = max(best, sum(f[i][j] for i, j in cells))
    return best

for f, exp in examples:
    print(max_fruits_brute(f), '| kỳ vọng', exp)

100 | kỳ vọng 100
4 | kỳ vọng 4


**Độ phức tạp:** `O(9^n · n)` — chỉ cho `n ≤ 5`.

## 6. Ý tưởng thuật toán / bước đột phá

`Đáp án = Σ f[i][i] + best(f) + best(fᵀ)`, trong đó `best(g)` = tổng lớn nhất trẻ 2 thu được trên các ô `j > i` (không tính ô đích):

- `dp[0][n−1] = g[0][n−1]`;
- `dp[i][j] = g[i][j] + max(dp[i−1][j−1], dp[i−1][j], dp[i−1][j+1])` với `j > i` (ô ngoài miền hoặc chưa đến được = `−∞`);
- `best = dp[n−2][n−1]` (bước cuối từ `(n−2, n−1)` vào đích — ô `(n−2, n−2)` nằm trên đường chéo).

Trẻ 3 trên lưới gốc ⇔ trẻ 2 trên lưới chuyển vị.

## 7. Các cách tiếp theo

### 7.1. Cách 2 — DP top-down có nhớ cho trẻ 2

In [4]:
from functools import lru_cache

def max_fruits_memo(f):
    n = len(f)
    NEG = float('-inf')
    def best(g):
        @lru_cache(maxsize=None)
        def go(i, j):                       # tổng tốt nhất từ (i, j) tới (n-2, n-1), đã tính (i, j)
            if not (i < j < n):
                return NEG
            if i == n - 2:
                return g[i][j] if j == n - 1 else NEG
            return g[i][j] + max(go(i + 1, j - 1), go(i + 1, j), go(i + 1, j + 1))
        r = go(0, n - 1) if n > 2 else g[0][n - 1]
        go.cache_clear()
        return r
    t = [list(r) for r in zip(*f)]
    return sum(f[i][i] for i in range(n)) + best(f) + best(t)

for f, exp in examples:
    print(max_fruits_memo(f), '| kỳ vọng', exp)

100 | kỳ vọng 100
4 | kỳ vọng 4


### 7.2. Cách 3 — DP lặp theo hàng (lời giải chính)

In [5]:
def child_best(g):
    n = len(g); NEG = float('-inf')
    cur = [NEG] * n
    cur[n - 1] = g[0][n - 1]
    for i in range(1, n - 1):
        nxt = [NEG] * n
        row = g[i]
        for j in range(i + 1, n):
            b = cur[j]
            if j - 1 > i - 1 and cur[j - 1] > b:
                b = cur[j - 1]
            if j + 1 < n and cur[j + 1] > b:
                b = cur[j + 1]
            if b != NEG:
                nxt[j] = b + row[j]
        cur = nxt
    return cur[n - 1]

def max_fruits_dp(f):
    n = len(f)
    t = [list(r) for r in zip(*f)]
    return sum(f[i][i] for i in range(n)) + child_best(f) + child_best(t)

for f, exp in examples:
    print(max_fruits_dp(f), '| kỳ vọng', exp)

100 | kỳ vọng 100
4 | kỳ vọng 4


### 7.3. Cách 4 — Vector hóa theo hàng bằng `numpy`

Mỗi hàng: `nxt = row + max(shift trái, giữa, shift phải)` của hàng trước, rồi đặt `−∞` cho cột `≤ i`.

In [6]:
import numpy as np

def child_best_np(g):
    g = np.asarray(g, dtype=np.int64); n = len(g)
    NEG = np.int64(-10 ** 15)
    cur = np.full(n, NEG); cur[n - 1] = g[0, n - 1]
    cols = np.arange(n)
    for i in range(1, n - 1):
        left = np.concatenate(([NEG], cur[:-1]))
        right = np.concatenate((cur[1:], [NEG]))
        b = np.maximum(np.maximum(left, cur), right)
        nxt = np.where((cols > i) & (b > NEG // 2), b + g[i], NEG)
        cur = nxt
    return int(cur[n - 1])

def max_fruits_np(f):
    a = np.asarray(f, dtype=np.int64)
    return int(np.trace(a)) + child_best_np(a) + child_best_np(a.T)

for f, exp in examples:
    print(max_fruits_np(f), '| kỳ vọng', exp)

100 | kỳ vọng 100
4 | kỳ vọng 4


## 8. TRACE từng bước

Trace DP trẻ 2 trên ví dụ 1 (in hàng `dp` sau mỗi hàng).

In [7]:
def trace_child(g):
    n = len(g); NEG = float('-inf')
    cur = [NEG] * n; cur[n - 1] = g[0][n - 1]
    print('hàng 0:', ['-' if x == NEG else x for x in cur])
    for i in range(1, n - 1):
        nxt = [NEG] * n
        for j in range(i + 1, n):
            cand = [cur[j]] + ([cur[j - 1]] if j - 1 > i - 1 else []) + ([cur[j + 1]] if j + 1 < n else [])
            if max(cand) != NEG:
                nxt[j] = max(cand) + g[i][j]
        cur = nxt
        print(f'hàng {i}:', ['-' if x == NEG else x for x in cur])
    print('best =', cur[n - 1])

f = examples[0][0]
print('trẻ 2:'); trace_child(f)
print('trẻ 3 (chuyển vị):'); trace_child([list(r) for r in zip(*f)])
print('đường chéo:', sum(f[i][i] for i in range(len(f))))

trẻ 2:
hàng 0: ['-', '-', '-', 4]
hàng 1: ['-', '-', 12, 11]
hàng 2: ['-', '-', '-', 24]
best = 24
trẻ 3 (chuyển vị):
hàng 0: ['-', '-', '-', 13]
hàng 1: ['-', '-', 23, 27]
hàng 2: ['-', '-', '-', 42]
best = 42
đường chéo: 34


## 9. Code nộp LeetCode

In [8]:
class Solution:
    def maxCollectedFruits(self, fruits: List[List[int]]) -> int:
        n = len(fruits)
        NEG = float('-inf')

        def best(g):
            cur = [NEG] * n
            cur[n - 1] = g[0][n - 1]
            for i in range(1, n - 1):
                nxt = [NEG] * n
                row = g[i]
                for j in range(max(i + 1, n - 1 - i), n):
                    b = cur[j]
                    if j - 1 > i - 1 and cur[j - 1] > b:
                        b = cur[j - 1]
                    if j + 1 < n and cur[j + 1] > b:
                        b = cur[j + 1]
                    if b != NEG:
                        nxt[j] = b + row[j]
                cur = nxt
            return cur[n - 1]

        t = [list(r) for r in zip(*fruits)]
        return sum(fruits[i][i] for i in range(n)) + best(fruits) + best(t)

for f, exp in examples:
    got = Solution().maxCollectedFruits(f)
    print(got, 'OK' if got == exp else 'SAI')

100 OK
4 OK


Bản nộp còn bỏ qua các cột `j < n − 1 − i` (không thể đến từ `(0, n−1)` sau `i` bước vì mỗi bước cột giảm tối đa 1) — cắt một nửa công việc.

## 10. Vì sao thuật toán đúng

1. **Trẻ 1** bị ép đi đường chéo (3.1) ⇒ lấy hết quả trên đường chéo.
2. **Trẻ 2 trên đường chéo hoặc dưới đường chéo không có lợi:** ô đường chéo đã rỗng; nếu trẻ 2 đi xuống dưới đường chéo, đoạn đó có thể "phản chiếu" lên trên (đổi `(i, j)` thành `(j, i)`… — trên thực tế, với vai trò đối xứng của trẻ 3, mọi ô dưới đường chéo mà trẻ 2 lấy cũng có thể để trẻ 3 lấy). Vét cạn mọi cặp đường (mục 12) xác nhận tổng tối ưu đạt được khi trẻ 2 ở trên và trẻ 3 ở dưới đường chéo.
3. **Tách độc lập:** khi trẻ 2 chỉ ở `j > i` và trẻ 3 chỉ ở `j < i`, ba tập ô rời nhau (trừ đích) ⇒ tổng = tổng ba phần, tối ưu từng phần bằng DP.
4. **DP trẻ 2** đúng theo quy nạp hàng (mỗi bước tăng hàng 1, cột thay đổi `±1/0`).

## 11. Phân tích Time / Space Complexity

| Cách | Time | Space |
|---|---|---|
| 1. Mọi cặp đường | `O(9^n · n)` | `O(3^n · n)` |
| 2. Top-down có nhớ | `O(n²)` | `O(n²)` |
| 3. DP lặp theo hàng | `O(n²)` | `O(n)` |
| 4. `numpy` theo hàng | `O(n²)` (vector hóa) | `O(n)` |

## 12. Bộ test

### 12.1. Ví dụ đề + ca biên

In [9]:
impls = {'memo': max_fruits_memo, 'dp': max_fruits_dp, 'numpy': max_fruits_np,
         'submit': lambda f: Solution().maxCollectedFruits(f)}
edge = [([[0, 0], [0, 0]], 0), ([[5, 9], [7, 1]], 22), ([[1, 0, 9], [0, 1, 0], [9, 0, 1]], 21), ([[0] * 3 for _ in range(3)], 0)]
for f, exp in examples + edge:
    assert max_fruits_brute(f) == exp, (f, max_fruits_brute(f))
    for name, fn in impls.items():
        assert fn(f) == exp, (name, f)
print('Ví dụ đề và ca biên: tất cả đúng.')

Ví dụ đề và ca biên: tất cả đúng.


### 12.2. Đối chiếu ngẫu nhiên với vét cạn

In [10]:
import random
rng = random.Random(3363)
cnt = 0
for _ in range(250):
    n = rng.randint(2, 5)
    f = [[rng.randint(0, rng.choice([3, 20, 1000])) for _ in range(n)] for _ in range(n)]
    exp = max_fruits_brute(f)
    for name, fn in impls.items():
        assert fn(f) == exp, (name, f)
    cnt += 1
for _ in range(100):
    n = rng.randint(2, 40)
    f = [[rng.randint(0, 1000) for _ in range(n)] for _ in range(n)]
    assert max_fruits_memo(f) == max_fruits_dp(f) == max_fruits_np(f) == Solution().maxCollectedFruits(f)
print('Khớp vét cạn trên', cnt, 'lưới nhỏ; 4 cách khớp nhau trên 100 lưới vừa.')

Khớp vét cạn trên 250 lưới nhỏ; 4 cách khớp nhau trên 100 lưới vừa.


### 12.3. Ca lớn `n = 1000`

In [11]:
import time
n = 1000
cases = {
    'ngẫu nhiên': [[rng.randint(0, 1000) for _ in range(n)] for _ in range(n)],
    'toàn 1000': [[1000] * n for _ in range(n)],
    'chỉ đường chéo phụ có quả': [[1000 if i + j == n - 1 else 0 for j in range(n)] for i in range(n)],
}
for name, f in cases.items():
    t = time.time(); a = Solution().maxCollectedFruits(f); t1 = time.time() - t
    t = time.time(); b = max_fruits_np(f); t2 = time.time() - t
    assert a == b
    print(f"{name:26s}: {a:9d} | thuần Python {t1:.2f}s | numpy {t2:.2f}s")

ngẫu nhiên                :   2109518 | thuần Python 0.15s | numpy 0.05s
toàn 1000                 :   2998000 | thuần Python 0.10s | numpy 0.05s


chỉ đường chéo phụ có quả :   1000000 | thuần Python 0.11s | numpy 0.04s


## 13. Các lỗi người mới hay gặp

### Lỗi 1 — Cho trẻ 2/3 đi tự do và cộng quả **không** loại trùng

In [12]:
def wrong_no_dedup(f):
    n = len(f); NEG = float('-inf')
    def free_best(g):
        cur = [NEG] * n; cur[n - 1] = g[0][n - 1]
        for i in range(1, n):
            nxt = [NEG] * n
            for j in range(n):
                b = max(cur[max(j - 1, 0):j + 2])
                if b != NEG:
                    nxt[j] = b + g[i][j]
            cur = nxt
        return cur[n - 1]
    t = [list(r) for r in zip(*f)]
    return sum(f[i][i] for i in range(n)) + free_best(f) + free_best(t)    # SAI: ô đích, ô chung tính nhiều lần

print('sai :', wrong_no_dedup(examples[1][0]), '| đúng:', Solution().maxCollectedFruits(examples[1][0]))

sai : 6 | đúng: 4


**Vì sao sai:** ô đích `(n−1, n−1)` được cả ba trẻ "lấy", và trẻ 2/3 có thể đi qua ô đường chéo đã rỗng. Quả mỗi ô chỉ tính một lần.

### Lỗi 2 — Cho trẻ 1 đi tự do như một DP thứ ba

In [13]:
def wrong_child1_free(f):
    n = len(f)
    # SAI: coi trẻ 1 cũng có thể chọn đường; ở đây lấy max đường từ (0,0) tới đích với 3 kiểu bước, không giới hạn số bước
    best = [[0] * n for _ in range(n)]
    for i in range(n):
        for j in range(n):
            prev = [best[i - 1][j] if i else None, best[i][j - 1] if j else None, best[i - 1][j - 1] if i and j else None]
            prev = [p for p in prev if p is not None]
            best[i][j] = f[i][j] + (max(prev) if prev else 0)
    return best[n - 1][n - 1] + child_best(f) + child_best([list(r) for r in zip(*f)]) - f[n - 1][n - 1]

print('sai :', wrong_child1_free(examples[0][0]), '| đúng:', Solution().maxCollectedFruits(examples[0][0]))

sai : 123 | đúng: 100


**Vì sao sai:** trẻ 1 phải đi **đúng** `n − 1` bước nên chỉ có một đường (đường chéo); cho đi tự do vừa sai luật vừa đếm trùng với trẻ 2/3.

### Lỗi 3 — Cho trẻ 2 đứng trên đường chéo và cộng lại quả ở đó

In [14]:
def wrong_allow_diag(f):
    n = len(f); NEG = float('-inf')
    def best(g):
        cur = [NEG] * n; cur[n - 1] = g[0][n - 1]
        for i in range(1, n - 1):
            nxt = [NEG] * n
            for j in range(i, n):                               # SAI: cho j = i (ô đường chéo)
                b = max(cur[j - 1] if j - 1 >= i - 1 else NEG, cur[j], cur[j + 1] if j + 1 < n else NEG)
                if b != NEG:
                    nxt[j] = b + g[i][j]                        # và cộng quả đường chéo lần nữa
            cur = nxt
        return max(cur[n - 1], cur[n - 2] if n >= 2 else NEG)
    t = [list(r) for r in zip(*f)]
    return sum(f[i][i] for i in range(n)) + best(f) + best(t)

rng = random.Random(4)
ce = next(f for f in ([[rng.randint(0, 9) for _ in range(4)] for _ in range(4)] for _ in range(5000))
          if wrong_allow_diag(f) != max_fruits_brute(f))
print('phản ví dụ:', ce, '| sai:', wrong_allow_diag(ce), '| đúng:', max_fruits_brute(ce))

phản ví dụ: [[3, 4, 1, 6], [7, 2, 1, 1], [0, 6, 8, 4], [0, 3, 8, 8]] | sai: 50 | đúng: 46


**Vì sao sai:** quả trên đường chéo đã thuộc trẻ 1; cho trẻ 2 (hoặc 3) "lấy lại" làm đếm trùng — kết quả lớn hơn thực tế.

### Lỗi 4 — Dùng cùng ma trận cho trẻ 3 (quên chuyển vị)

In [15]:
def wrong_no_transpose(f):
    n = len(f)
    return sum(f[i][i] for i in range(n)) + 2 * child_best(f)

print('sai :', wrong_no_transpose(examples[0][0]), '| đúng:', Solution().maxCollectedFruits(examples[0][0]))

sai : 82 | đúng: 100


**Vì sao sai:** trẻ 3 đi ở tam giác **dưới** đường chéo; phải chạy DP trên `fᵀ`.

## 14. Mẹo làm bài nhanh & nhận dạng pattern

- "Đi đúng `k` bước tới đích" ⇒ kiểm tra xem đường có bị ép không.
- Nhiều tác nhân tránh nhau ⇒ tìm "vùng" riêng cho mỗi tác nhân để tách DP độc lập.
- Đối xứng qua đường chéo ⇒ chuyển vị để dùng lại cùng một hàm.
- Đánh dấu ô không đến được bằng `−∞`.

## 15. Biến thể của bài

1. **LC 1463 — Cherry Pickup II:** hai robot cùng đi xuống ⇒ DP trạng thái hai cột.
2. **LC 741 — Cherry Pickup:** đi rồi về ⇒ DP hai người đi đồng thời.
3. Bốn trẻ (thêm góc `(n−1, n−1)` đi ngược về `(0,0)`) ⇒ phân vùng khác.
4. Trẻ 2 được đi đúng `n − 1` bước nhưng đích bất kỳ ở hàng cuối ⇒ bỏ điều kiện cột cuối.

In [16]:
# Biến thể 1 (LC 1463): hai robot từ (0,0) và (0,cols-1), đi xuống, cùng ô chỉ tính một lần
def cherry_pickup_ii(grid):
    R, C = len(grid), len(grid[0]); NEG = float('-inf')
    dp = [[NEG] * C for _ in range(C)]
    dp[0][C - 1] = grid[0][0] + (grid[0][C - 1] if C > 1 else 0)
    for r in range(1, R):
        nd = [[NEG] * C for _ in range(C)]
        for a in range(C):
            for b in range(C):
                if dp[a][b] == NEG:
                    continue
                for da in (-1, 0, 1):
                    for db in (-1, 0, 1):
                        x, y = a + da, b + db
                        if 0 <= x < C and 0 <= y < C:
                            v = dp[a][b] + grid[r][x] + (grid[r][y] if x != y else 0)
                            if v > nd[x][y]:
                                nd[x][y] = v
        dp = nd
    return max(max(row) for row in dp)

assert cherry_pickup_ii([[3, 1, 1], [2, 5, 1], [1, 5, 5], [2, 1, 1]]) == 24
assert cherry_pickup_ii([[1, 0, 0, 0, 0, 0, 1], [2, 0, 0, 0, 0, 3, 0], [2, 0, 9, 0, 0, 0, 0], [0, 3, 0, 5, 4, 0, 0], [1, 0, 2, 3, 0, 0, 6]]) == 28
print('LC 1463 ví dụ: 24, 28 — đúng.')

LC 1463 ví dụ: 24, 28 — đúng.


## 16. So sánh các cách làm

| Cách | Ý tưởng | Time | Ghi chú |
|---|---|---|---|
| 1 | Mọi cặp đường | `O(9^n)` | Oracle |
| 2 | Top-down có nhớ | `O(n²)` | Gọn |
| 3 | DP lặp theo hàng | `O(n²)` | **Nộp bài** |
| 4 | `numpy` | `O(n²)` | Nhanh nhất |

## 17. 5 bài tập vận dụng (từ dễ đến khó)

**Bài 1 (đoán output):** lưới `2 × 2` `[[3, 4], [5, 6]]`.

**Bài 2 (tính tay):** lưới `3 × 3` toàn số 1.

**Bài 3 (điền TODO):** dựng **đường đi** của trẻ 2 đạt tối ưu.

**Bài 4 (tự viết lại):** đếm **số đường đi** của trẻ 2 (trong tam giác `j > i`) đạt tổng tối ưu.

**Bài 5 (mini challenge):** kiểm chứng bằng vét cạn rằng giả định "trẻ 2 ở trên, trẻ 3 ở dưới đường chéo" không làm mất tối ưu với mọi lưới nhị phân `4 × 4`.

In [17]:
def child2_path_todo(f):
    # TODO
    raise NotImplementedError

def count_optimal_child2_todo(f):
    # TODO
    raise NotImplementedError

def verify_split_all_binary_todo():
    # TODO
    raise NotImplementedError

print('Đã định nghĩa các hàm TODO (không gọi).')

Đã định nghĩa các hàm TODO (không gọi).


## 18. Gợi ý 3 mức cho bài tập

**Bài 1.** (1) Ba trẻ, mỗi trẻ 1 bước. (2) Mọi ô được thăm. (3) `18`.

**Bài 2.** (1) Đường chéo 3. (2) Trẻ 2: `(0,2) → (1,2)` = 2. (3) Trẻ 3 đối xứng 2 ⇒ `7`.

**Bài 3.** (1) Lưu cột cha ở mỗi ô. (2) Truy vết từ `(n−2, n−1)`. (3) Thêm đích.

**Bài 4.** (1) DP lưu cặp `(giá trị tốt nhất, số đường)`. (2) Khi hai ô cha cùng giá trị tốt nhất, cộng số đường. (3) Vét cạn bằng `child2_paths` lọc `j > i`.

**Bài 5.** (1) `2^16` lưới. (2) So `max_fruits_brute` với DP. (3) Có thể chạy trên một mẫu ngẫu nhiên lớn nếu muốn nhanh.

## 19. Đáp án tham khảo cho bài tập

In [18]:
# Bài 1, 2
assert Solution().maxCollectedFruits([[3, 4], [5, 6]]) == 18
assert Solution().maxCollectedFruits([[1] * 3 for _ in range(3)]) == 7 == max_fruits_brute([[1] * 3 for _ in range(3)])
print('Bài 1: 18 | Bài 2: 7')

# Bài 3
def child2_path(f):
    n = len(f); NEG = float('-inf')
    dp = [[NEG] * n for _ in range(n)]; par = [[None] * n for _ in range(n)]
    dp[0][n - 1] = f[0][n - 1]
    for i in range(1, n - 1):
        for j in range(i + 1, n):
            for pj in (j - 1, j, j + 1):
                if 0 <= pj < n and pj > i - 1 and dp[i - 1][pj] != NEG and dp[i - 1][pj] + f[i][j] > dp[i][j]:
                    dp[i][j] = dp[i - 1][pj] + f[i][j]; par[i][j] = pj
    path, j = [(n - 1, n - 1)], n - 1
    for i in range(n - 2, -1, -1):
        path.append((i, j)); j = par[i][j] if i else j
    return dp[n - 2][n - 1], path[::-1]

val, path = child2_path(examples[0][0])
assert val == child_best(examples[0][0])
print('Bài 3:', val, path)

Bài 1: 18 | Bài 2: 7
Bài 3: 24 [(0, 3), (1, 2), (2, 3), (3, 3)]


In [19]:
# Bài 4, 5
def count_optimal_child2(f):
    n = len(f); NEG = float('-inf')
    cur = [(NEG, 0)] * n; cur[n - 1] = (f[0][n - 1], 1)
    for i in range(1, n - 1):
        nxt = [(NEG, 0)] * n
        for j in range(i + 1, n):
            best, ways = NEG, 0
            for pj in (j - 1, j, j + 1):
                if 0 <= pj < n and pj > i - 1 and cur[pj][0] != NEG:
                    if cur[pj][0] > best:
                        best, ways = cur[pj]
                    elif cur[pj][0] == best:
                        ways += cur[pj][1]
            if best != NEG:
                nxt[j] = (best + f[i][j], ways)
        cur = nxt
    return cur[n - 1]

def count_optimal_brute(f):
    n = len(f)
    vals = [sum(f[i][j] for i, j in p[:-1]) for p in child2_paths(n) if all(j > i for i, j in p[:-1])]
    return max(vals), vals.count(max(vals))

rng = random.Random(44)
for _ in range(200):
    n = rng.randint(2, 6)
    f = [[rng.randint(0, 2) for _ in range(n)] for _ in range(n)]
    assert count_optimal_child2(f) == count_optimal_brute(f)

def verify_split_sample(trials=200, n=4):
    rng = random.Random(55)
    for _ in range(trials):
        f = [[rng.randint(0, 1) for _ in range(n)] for _ in range(n)]
        assert max_fruits_brute(f) == max_fruits_dp(f)
    return True

print('Bài 4 (ví dụ 1: giá trị, số đường):', count_optimal_child2(examples[0][0]), '| Bài 5 (200 lưới nhị phân 4x4):', verify_split_sample())

Bài 4 (ví dụ 1: giá trị, số đường): (24, 1) | Bài 5 (200 lưới nhị phân 4x4): True


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

```python
ans = Σ f[i][i]                              # trẻ 1 bị ép đi đường chéo
def best(g):                                 # trẻ 2 trên tam giác j > i
    cur[n-1] = g[0][n-1]
    for i in 1..n-2: nxt[j] = g[i][j] + max(cur[j-1], cur[j], cur[j+1])  (j > i, -inf nếu không tới)
    return cur[n-1]
ans += best(f) + best(transpose(f))
```

**Checklist:**

- [ ] Trẻ 1 chỉ đi đường chéo?
- [ ] Trẻ 2 giữ `j > i`, kết thúc ở `(n−2, n−1)` rồi vào đích?
- [ ] Trẻ 3 = trẻ 2 trên ma trận chuyển vị?
- [ ] Ô đích chỉ tính một lần (trong tổng đường chéo)?
- [ ] Ô không đến được là `−∞`?